### 베이스라인 데이터 불러오기

In [ ]:
# ==========================================
# 라이브러리 임포트 (본인이 쓰는 것만 - 통일 강제 안 함)
# ==========================================
import os
from dotenv import load_dotenv, find_dotenv

# .env 파일을 명시적으로 로드 (다른 컴퓨터에서 실행할 때도
# OS 환경변수 설정 없이 리포지토리 루트의 .env 값을 그대로 읽어오기 위함)
_dotenv_path = find_dotenv()
load_dotenv(_dotenv_path)

# Windows 계정명에 한글 등 non-ASCII 문자가 섞여 있으면 OS 기본 임시 폴더
# (예: C:\Users\<계정명>\AppData\Local\Temp) 경로도 non-ASCII가 되고,
# joblib(loky)이 멀티프로세싱 리소스 트래커에서 그 경로를 ASCII로 인코딩하려다
# UnicodeEncodeError를 낸다. 리포지토리 하위의 ASCII 전용 경로를 joblib
# 임시 폴더로 강제 지정해 회피한다 (n_jobs=-1을 쓰는 Voting에 필요).
if _dotenv_path:
    _repo_root = os.path.dirname(_dotenv_path)
else:
    _repo_root = os.getcwd()
_joblib_temp_dir = os.path.join(_repo_root, ".joblib_tmp")
os.makedirs(_joblib_temp_dir, exist_ok=True)
os.environ.setdefault("JOBLIB_TEMP_FOLDER", _joblib_temp_dir)

import mlflow
import mlflow.artifacts
import numpy as np
import pandas as pd
import joblib
from datetime import datetime
from IPython.display import display
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ==========================================
# 경로 및 데이터 로드
# ==========================================
load_dir = os.path.join("D:/seoul_bike/models_pkl/train_pkl")

try:
    X_train = joblib.load(os.path.join(load_dir, "X_train.pkl"))
    Y_train = joblib.load(os.path.join(load_dir, "Y_train.pkl"))
    X_val = joblib.load(os.path.join(load_dir, "X_val.pkl"))
    Y_val = joblib.load(os.path.join(load_dir, "Y_val.pkl"))
    X_test = joblib.load(os.path.join(load_dir, "X_test.pkl"))
    Y_test = joblib.load(os.path.join(load_dir, "Y_test.pkl"))

    print(f"========== 데이터 로드 완료 ==========")
    print(f"학습 데이터 크기: {X_train.shape}")
    print(f"검증 데이터 크기: {X_val.shape}")
    print("※ X_test/Y_test는 최종 후보 선정 전까지 절대 사용하지 않습니다 (Val로만 비교·선택).")

except FileNotFoundError as e:
    print(f"파일을 찾을 수 없습니다. 경로를 확인해주세요: {e}")

### 앙상블 파이프라인 (Voting 필수 + Stacking 확장)

* `EXPERIMENT_NAME`: `bike_demand_prediction_ensemble` (팀 공통 고정)
* 하이퍼파라미터는 직접 입력하지 않고 **MLflow API**로 튜닝 실험(`bike_demand_prediction_tuning`)의 최신 run에 로깅된 `best_params`를 조회해 그대로 사용
  (아티팩트 실체는 Supabase Storage `mlflow-artifacts` 버킷에 `experiment_id/run_id/artifacts/...`로 저장되어 있고, MLflow가 S3 호환 자격증명으로 그 경로를 알아서 찾아준다 - supabase-py로 경로를 직접 조립하는 것보다 안전)
* Base 모델: LightGBM / XGBoost (Voting·Stacking 공통), Final Estimator(메타모델): Ridge 고정
* Voting은 기준선으로 필수 제출, Stacking은 `TimeSeriesSplit(n_splits=5)` 기반 OOF로 Data Leakage 차단 후 확장 비교용

In [ ]:
# ==========================================
# 환경 설정 (AUTHOR / EXPERIMENT_NAME / MLflow·Supabase 연동)
# ==========================================
AUTHOR = "장수연"
EXPERIMENT_NAME = "bike_demand_prediction_ensemble"  # 팀 전체 결과가 한 실험에 모여야 비교 가능 (고정값)

SEED = 42
N_SPLITS = 5  # 팀 공통값: 튜닝 단계에서 쓴 N_SPLITS와 동일하게 유지 (폴드 조건 달라지면 튜닝 결과와 비교 불가)

# MLflow 연동
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"========== MLflow 연동 완료: {MLFLOW_TRACKING_URI} ==========")

# Supabase Storage 연동 (MLflow 아티팩트가 실제로 저장되는 S3 호환 백엔드)
# - mlflow.artifacts.load_dict()가 run의 아티팩트를 내려받을 때 boto3를 통해
#   이 자격증명으로 Supabase Storage(mlflow-artifacts 버킷)에 접근한다.
if os.getenv("SUPABASE_S3_ENDPOINT_URL"):
    os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("SUPABASE_S3_ENDPOINT_URL")
    os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("SUPABASE_ACCESS_KEY_ID", "")
    os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("SUPABASE_SECRET_ACCESS_KEY", "")
    os.environ["AWS_REGION"] = os.getenv("SUPABASE_REGION", "ap-northeast-2")
    print("========== Supabase Storage 연동 완료 ==========")

mlflow.set_experiment(EXPERIMENT_NAME)


# ==========================================
# 평가 지표 함수 (baseline / tuning과 이름·계산식 동일 - 실험 간 비교 가능해야 함)
# ==========================================
def rmsle(y_true_raw, pred_raw):
    log_y = np.log1p(np.maximum(y_true_raw, 0))
    log_pred = np.log1p(np.maximum(pred_raw, 0))
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def evaluate_regr(y_true_raw, pred_raw):
    return {
        "rmsle": rmsle(y_true_raw, pred_raw),
        "rmse": np.sqrt(mean_squared_error(y_true_raw, pred_raw)),
        "mae": mean_absolute_error(y_true_raw, pred_raw)
    }

# 학습은 log1p(y)로, 예측 후 expm1 + 음수 clip으로 복원 (TransformedTargetRegressor에서 사용)
def inverse_log_clip(x):
    return np.clip(np.expm1(x), 0, None)


# ==========================================
# [Step 1] MLflow API로 best_params 불러오기 (Supabase Storage 백엔드)
# ==========================================
# 하이퍼파라미터는 손으로 넣지 않고, tuning 단계에서 mlflow.log_dict(best_params, ...)로
# 저장한 값을 타깃 x 모델별로 조회해서 그대로 사용한다.
# 주의: 튜닝을 여러 번 재실행하면서 run 이름에 '_2', '_3' 같은 접미사가 붙기 때문에
# 아티팩트 파일명이 '{run_name}_best_params.json' 형태로 매번 달라진다
# (예: 'LightGBM_general_rent_cnt_3_best_params.json').
# 그래서 파일명을 '{model}_{target}_best_params.json'으로 고정 조립하지 않고,
# 찾은 run의 아티팩트 목록에서 '_best_params.json'으로 끝나는 파일을 그대로 찾아 로드한다.
TUNING_EXPERIMENT_NAME = "bike_demand_prediction_tuning"
_best_params_cache = {}

def get_best_params_from_mlflow(model_name, target_name, experiment_name=TUNING_EXPERIMENT_NAME):
    """
    tuning 실험에서 (model, target) 태그가 일치하는 가장 최근 run을 찾아
    그 run의 아티팩트 중 '_best_params.json'으로 끝나는 파일을 로드한다.
    - 조합을 못 찾으면 임의로 채우지 않고, 콘솔에 경고를 출력한 뒤 None을 반환한다.
      (호출부에서 None이면 디폴트 파라미터를 명시적으로 사용)
    """
    cache_key = (model_name, target_name)
    if cache_key in _best_params_cache:
        return _best_params_cache[cache_key]

    best_params = None
    try:
        experiment = mlflow.get_experiment_by_name(experiment_name)
        if experiment is None:
            print(f"[경고] MLflow 실험 '{experiment_name}'을 찾을 수 없습니다. ({model_name}/{target_name} -> 디폴트 파라미터 사용)")
        else:
            runs = mlflow.search_runs(
                experiment_ids=[experiment.experiment_id],
                filter_string=f"tags.model = '{model_name}' and tags.target = '{target_name}'",
                order_by=["start_time DESC"],
                max_results=1,
            )

            if runs.empty:
                print(f"[경고] '{model_name}' / '{target_name}' 조합의 튜닝 Run을 찾을 수 없습니다. -> 디폴트 파라미터 사용")
            else:
                run_id = runs.iloc[0]["run_id"]
                artifact_files = mlflow.artifacts.list_artifacts(run_id=run_id)
                best_params_files = [f.path for f in artifact_files if f.path.endswith("_best_params.json")]

                if not best_params_files:
                    print(f"[경고] run_id={run_id}에 '_best_params.json' 아티팩트가 없습니다. -> 디폴트 파라미터 사용")
                else:
                    artifact_path = best_params_files[0]
                    best_params = mlflow.artifacts.load_dict(f"runs:/{run_id}/{artifact_path}")
                    print(f"[로드 완료] '{model_name}' / '{target_name}' best_params <- run_id={run_id}, file={artifact_path}")

    except Exception as e:
        print(f"[경고] '{model_name}' / '{target_name}' best_params 로드 실패: {e} -> 디폴트 파라미터 사용")
        best_params = None

    _best_params_cache[cache_key] = best_params
    return best_params


# ==========================================
# Base 모델 빌더
# ==========================================
# 앙상블 내부(Voting/Stacking)에서 n_jobs=-1로 병렬처리하므로,
# 개별 Base 모델의 n_jobs는 1로 제한해 프로세스 경합을 방지한다.
def build_base_model(model_name, target_name, SEED=SEED):
    best_params = get_best_params_from_mlflow(model_name, target_name)
    params = best_params if best_params else {}

    if model_name == "LightGBM":
        return LGBMRegressor(random_state=SEED, n_jobs=1, verbosity=-1, **params)
    elif model_name == "XGBoost":
        return XGBRegressor(random_state=SEED, n_jobs=1, tree_method='hist', **params)
    else:
        raise ValueError(f"알 수 없는 모델명: {model_name}")


# ==========================================
# 수동 OOF Stacking (TimeSeriesSplit 대응)
# ==========================================
# sklearn의 StackingRegressor는 내부적으로 cross_val_predict()를 쓰는데, 이 함수는
# "test fold를 다 합치면 전체 샘플을 정확히 한 번씩 커버해야 한다(partition)"는 조건을
# 강제한다. 그런데 TimeSeriesSplit은 맨 앞 구간(첫 폴드의 학습 전용 데이터)을 시계열
# 특성상 어떤 test fold에도 포함시키지 않으므로(그 이전 데이터가 없어 OOF 예측 자체가
# 불가능) 이 조건을 만족하지 못해 "cross_val_predict only works for partitions"
# ValueError가 발생한다. 그래서 StackingRegressor 대신 TimeSeriesSplit 폴드를 직접
# 순회하며 OOF 예측을 만들고, OOF가 없는 맨 앞 구간은 메타모델 학습에서 제외한다.
class ManualStackingRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_model_names, target_name, final_estimator, cv):
        self.base_model_names = base_model_names
        self.target_name = target_name
        self.final_estimator = final_estimator
        self.cv = cv

    def fit(self, X, y):
        X = X.reset_index(drop=True)
        y = np.asarray(y)

        # 1) 폴드별로 base 모델을 새로 학습시켜 OOF 예측 수집
        oof_preds = {name: np.full(len(X), np.nan) for name in self.base_model_names}
        for name in self.base_model_names:
            for train_idx, test_idx in self.cv.split(X):
                fold_model = build_base_model(name, self.target_name)
                fold_model.fit(X.iloc[train_idx], y[train_idx])
                oof_preds[name][test_idx] = fold_model.predict(X.iloc[test_idx])

        # 2) OOF가 없는(맨 앞 구간) 샘플은 메타모델 학습에서 제외
        oof_df = pd.DataFrame(oof_preds)
        valid_mask = oof_df.notna().all(axis=1)
        self.final_estimator_ = clone(self.final_estimator)
        self.final_estimator_.fit(oof_df[valid_mask], y[valid_mask.values])

        # 3) 예측 시 사용할 base 모델은 전체 학습 데이터로 다시 학습
        self.final_base_models_ = {
            name: build_base_model(name, self.target_name).fit(X, y)
            for name in self.base_model_names
        }
        return self

    def predict(self, X):
        base_preds = pd.DataFrame({
            name: model.predict(X) for name, model in self.final_base_models_.items()
        })
        return self.final_estimator_.predict(base_preds)


# ==========================================
# [Step 2, 3] Voting -> Stacking 학습 루프 (타깃마다 순서대로 학습/평가/MLflow 로깅)
# ==========================================
print("\n========== 앙상블 모델 학습 시작 ==========")

ENSEMBLE_BASE_MODELS = ["LightGBM", "XGBoost"]  # Voting/Stacking 공통 Base Estimator
TARGET_COLUMNS = ['general_rent_cnt', 'sprout_rent_cnt', 'general_rtn_cnt', 'sprout_rtn_cnt']
RUN_STACKING = True  # Voting은 기준선으로 필수, Stacking은 확장 비교(필요 없으면 False)

run_date = datetime.now().strftime("%Y-%m-%d %H:%M")
results = []

for target_name in TARGET_COLUMNS:
    print(f"---------- [{target_name}] 앙상블 학습 시작 ----------")

    y_train_raw = Y_train[target_name]
    y_val_raw = Y_val[target_name]

    # ------------------------------------------
    # [Step 2] Voting Regressor (기준선 - 필수 제출)
    # ------------------------------------------
    # 타깃마다 Base Estimator를 새로 생성 (재사용 금지: fit() 시 이전 타깃 상태가 덮어써지는 것 방지)
    voting_base_estimators = [
        (m_name, build_base_model(m_name, target_name)) for m_name in ENSEMBLE_BASE_MODELS
    ]

    voting_core = VotingRegressor(estimators=voting_base_estimators, n_jobs=-1)

    # 모델 전체를 TransformedTargetRegressor로 감싸서 log1p 학습 / expm1 복원 자동화
    voting_model = TransformedTargetRegressor(
        regressor=voting_core,
        func=np.log1p,
        inverse_func=inverse_log_clip
    )

    voting_model.fit(X_train, y_train_raw)
    voting_pred = voting_model.predict(X_val)
    voting_metrics = evaluate_regr(y_val_raw.values, voting_pred)

    with mlflow.start_run(run_name=f"Voting_{target_name}"):
        mlflow.set_tag("target", target_name)
        mlflow.set_tag("model", "Voting")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(voting_metrics)
        # skops(mlflow.sklearn.log_model 기본 직렬화)는 커스텀 함수(inverse_log_clip)와
        # LightGBM/XGBoost Booster 객체를 "신뢰되지 않은 타입"으로 보고 로드를 거부하므로
        # cloudpickle 방식으로 명시 지정한다.
        mlflow.sklearn.log_model(voting_model, name="model", serialization_format="cloudpickle")

    results.append({
        "target": target_name, "model_name": "Voting",
        "RMSLE": voting_metrics["rmsle"], "RMSE": voting_metrics["rmse"], "MAE": voting_metrics["mae"]
    })
    print(f"[Voting_{target_name}] RMSLE={voting_metrics['rmsle']:.4f}")

    # ------------------------------------------
    # [Step 3] Stacking (수동 OOF, 확장 - 비교 대상)
    # ------------------------------------------
    if RUN_STACKING:
        stacking_core = ManualStackingRegressor(
            base_model_names=ENSEMBLE_BASE_MODELS,
            target_name=target_name,
            final_estimator=Ridge(),  # 메타모델은 Ridge로 고정
            cv=TimeSeriesSplit(n_splits=N_SPLITS),  # OOF 예측 - 시계열 누수 방지 (일반 KFold 금지)
        )

        stacking_model = TransformedTargetRegressor(
            regressor=stacking_core,
            func=np.log1p,
            inverse_func=inverse_log_clip
        )

        stacking_model.fit(X_train, y_train_raw)
        stacking_pred = stacking_model.predict(X_val)
        stacking_metrics = evaluate_regr(y_val_raw.values, stacking_pred)

        with mlflow.start_run(run_name=f"Stacking_{target_name}"):
            mlflow.set_tag("target", target_name)
            mlflow.set_tag("model", "Stacking")
            mlflow.set_tag("author", AUTHOR)
            mlflow.set_tag("run_date", run_date)
            mlflow.log_metrics(stacking_metrics)
            # Voting과 동일한 이유로 cloudpickle 직렬화 사용
            mlflow.sklearn.log_model(stacking_model, name="model", serialization_format="cloudpickle")

        results.append({
            "target": target_name, "model_name": "Stacking",
            "RMSLE": stacking_metrics["rmsle"], "RMSE": stacking_metrics["rmse"], "MAE": stacking_metrics["mae"]
        })
        print(f"[Stacking_{target_name}] RMSLE={stacking_metrics['rmsle']:.4f}")

print("\n========== 앙상블 모델 학습 완료 ==========")


# ==========================================
# [Step 4] 결과 비교 테이블 (baseline/tuning과 동일한 리포팅 포맷)
# ==========================================
results_df = pd.DataFrame(results)
display(results_df.sort_values(by=['target', 'RMSLE']))

### 타깃별 최종 챔피언 모델 선정 (RandomForest 후보 포함)

* 지금까지의 Voting/Stacking은 팀 규칙대로 LightGBM/XGBoost만 base로 썼는데, 튜닝 리더보드를 보면 `sprout_rtn_cnt`는 RandomForest가 단일 최고 성능이었다 (앙상블 base에 아예 없었으니 이길 수 없는 구조).
* 그래서 RandomForest까지 포함한 단일 모델 3종(LightGBM/XGBoost/RandomForest, 전부 튜닝된 best_params 그대로)을 이 노트북과 동일한 X_val 기준으로 다시 평가하고, 이미 계산된 Voting/Stacking 결과와 합쳐서 **타깃별로 검증 RMSLE가 가장 낮은 모델**을 챔피언으로 뽑는다.
* 챔피언 요약은 MLflow에 별도 run으로 태깅해서, 나중에 서빙할 모델을 타깃별로 바로 찾을 수 있게 한다.

In [ ]:
# ==========================================
# [Step 5] 단일 후보(LightGBM/XGBoost/RandomForest) 재평가
# ==========================================
# Voting/Stacking(results)에는 없는 RandomForest를 포함해, 3개 단일 모델을
# 전부 이 노트북의 X_train/X_val 기준으로 다시 학습/평가한다.
# (튜닝 리더보드 수치는 다른 실행에서 나온 값이라, 지금 로드된 데이터와
#  완전히 동일한 조건에서 비교하려면 여기서 한 번 더 재는 게 안전하다)
from sklearn.ensemble import RandomForestRegressor

SINGLE_CANDIDATE_MODELS = ["LightGBM", "XGBoost", "RandomForest"]

def build_single_model(model_name, target_name, SEED=SEED):
    best_params = get_best_params_from_mlflow(model_name, target_name)
    params = best_params if best_params else {}

    if model_name == "LightGBM":
        return LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1, **params)
    elif model_name == "XGBoost":
        return XGBRegressor(random_state=SEED, n_jobs=-1, tree_method='hist', **params)
    elif model_name == "RandomForest":
        return RandomForestRegressor(random_state=SEED, n_jobs=-1, **params)
    else:
        raise ValueError(f"알 수 없는 모델명: {model_name}")


print("\n========== 단일 후보 재평가 시작 (RandomForest 포함) ==========")

champion_results = list(results)  # Voting/Stacking 결과 재사용

for target_name in TARGET_COLUMNS:
    y_train_raw = Y_train[target_name]
    y_val_raw = Y_val[target_name]

    for m_name in SINGLE_CANDIDATE_MODELS:
        single_model = TransformedTargetRegressor(
            regressor=build_single_model(m_name, target_name),
            func=np.log1p,
            inverse_func=inverse_log_clip
        )
        single_model.fit(X_train, y_train_raw)
        single_pred = single_model.predict(X_val)
        single_metrics = evaluate_regr(y_val_raw.values, single_pred)

        champion_results.append({
            "target": target_name, "model_name": m_name,
            "RMSLE": single_metrics["rmsle"], "RMSE": single_metrics["rmse"], "MAE": single_metrics["mae"]
        })
        print(f"[{m_name}_{target_name}] RMSLE={single_metrics['rmsle']:.4f}")

print("\n========== 단일 후보 재평가 완료 ==========")


# ==========================================
# [Step 6] 타깃별 챔피언(RMSLE 최저) 선정 + MLflow 로깅
# ==========================================
champion_df = pd.DataFrame(champion_results)
champion_idx = champion_df.groupby("target")["RMSLE"].idxmin()
champion_table = champion_df.loc[champion_idx].sort_values("target").reset_index(drop=True)

print("\n========== 타깃별 최종 챔피언 ==========")
display(champion_table)

with mlflow.start_run(run_name="Champion_Summary"):
    mlflow.set_tag("author", AUTHOR)
    mlflow.set_tag("run_date", run_date)
    for _, row in champion_table.iterrows():
        mlflow.log_metric(f"{row['target']}_champion_rmsle", row["RMSLE"])
        mlflow.set_tag(f"{row['target']}_champion_model", row["model_name"])

print("\n========== 챔피언 요약 MLflow 로깅 완료 (run_name=Champion_Summary) ==========")

### 최종 테스트 (X_test, 딱 한 번만 사용)

* `champion_table`에서 뽑힌 타깃별 챔피언 모델을 `X_train`으로 다시 학습시키고, 지금까지 한 번도 쓰지 않은 `X_test`/`Y_test`로 최종 성능을 확인한다.
* 여기서 나온 결과를 보고 다시 튜닝하거나 모델을 바꾸면 X_test가 검증셋처럼 오염되므로, 이 결과는 최종 보고용으로만 사용한다.

In [ ]:
# ==========================================
# [Step 7] 최종 테스트 (X_test, 딱 한 번만 사용)
# ==========================================
# champion_table에서 뽑힌 타깃별 챔피언 모델 종류(Voting/Stacking/단일)를 그대로
# X_train으로 재학습시키고, 여태 한 번도 안 쓴 X_test로 최종 성능을 딱 한 번만 잰다.
print("\n========== 최종 테스트(X_test) 시작 ==========")

final_test_results = []

for _, champ in champion_table.iterrows():
    target_name = champ["target"]
    model_name = champ["model_name"]
    y_train_raw = Y_train[target_name]
    y_test_raw = Y_test[target_name]

    if model_name == "Stacking":
        core_model = ManualStackingRegressor(
            base_model_names=ENSEMBLE_BASE_MODELS,
            target_name=target_name,
            final_estimator=Ridge(),
            cv=TimeSeriesSplit(n_splits=N_SPLITS),
        )
    elif model_name == "Voting":
        core_model = VotingRegressor(
            estimators=[(m, build_base_model(m, target_name)) for m in ENSEMBLE_BASE_MODELS],
            n_jobs=-1
        )
    else:  # 단일 모델 (LightGBM / XGBoost / RandomForest)
        core_model = build_single_model(model_name, target_name)

    final_model = TransformedTargetRegressor(
        regressor=core_model,
        func=np.log1p,
        inverse_func=inverse_log_clip
    )
    final_model.fit(X_train, y_train_raw)
    test_pred = final_model.predict(X_test)
    test_metrics = evaluate_regr(y_test_raw.values, test_pred)

    with mlflow.start_run(run_name=f"FinalTest_{target_name}"):
        mlflow.set_tag("target", target_name)
        mlflow.set_tag("model", model_name)
        mlflow.set_tag("stage", "final_test")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(test_metrics)
        mlflow.sklearn.log_model(final_model, name="model", serialization_format="cloudpickle")

    final_test_results.append({
        "target": target_name, "champion_model": model_name,
        "RMSLE": test_metrics["rmsle"], "RMSE": test_metrics["rmse"], "MAE": test_metrics["mae"]
    })
    print(f"[FinalTest_{target_name}] model={model_name}, RMSLE={test_metrics['rmsle']:.4f}")

print("\n========== 최종 테스트 완료 ==========")

final_test_df = pd.DataFrame(final_test_results)
display(final_test_df)